merge the county-level election returns data from 
https://searchworks.stanford.edu/view/13803114 and 
https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/VOQCHQ


In [ ]:
import pandas as pd
import zipfile
import json
import numpy as np
import gzip
import os

# Load MIT data

In [ ]:
data_mit = pd.read_csv("../../../countypres_2000-2020.csv")

data_2000 = pd.read_csv("../../../2000_0_0_2.csv")

# Double check that MIT data line up with Dave Liep


In [ ]:
# prepare tables
data_2000 = pd.read_csv("../../../2000_0_0_2.csv")
data_2000 = data_2000[['FIPS', 'Total Vote', 'George W. Bush']]
data_2000 = data_2000.iloc[1:]
data_2000['FIPS'] = data_2000['FIPS'].astype(float)

data_2000_mit = data_mit[
    (data_mit['year'] == 2000) &
    (data_mit['party'] == 'REPUBLICAN')
]

# outer merge with indicator
merged = pd.merge(
    data_2000,
    data_2000_mit,
    left_on='FIPS',
    right_on='county_fips',
    how='outer',
    indicator=True
)

# FIPS present only in data_2000
left_only_fips = merged.loc[merged['_merge'] == 'left_only', 'FIPS']

# FIPS present only in data_2000_mit
right_only_fips = merged.loc[merged['_merge'] == 'right_only', 'county_fips']

print("FIPS only in data_2000:")
print(left_only_fips.dropna().unique())

print("\nFIPS only in data_2000_mit:")
print(right_only_fips.dropna().unique())

In [ ]:
(data_2000[data_2000['FIPS'] == 51560], data_2000[data_2000['FIPS'] == 51005])

In [ ]:
(data_2000_mit[data_2000_mit['county_fips'] == 51560], data_2000_mit[data_2000_mit['county_fips'] == 51005])

# Create county pres dataset

In [ ]:
prez = pd.read_csv('../../../countypres_2000-2020.csv')
prez = prez.groupby(['county_fips', 'party', 'state_po', 'year']).agg('sum').reset_index()
prez = prez[['county_fips', 'party', 'candidatevotes', 'totalvotes', 'state_po', 'year']]
prez = prez[(prez['party'] == 'REPUBLICAN')]
prez = prez.groupby(['county_fips', 'year']).agg('sum').reset_index()
prez['share_republican'] = prez['candidatevotes']/prez['totalvotes']
prez = prez[['county_fips', 'share_republican', 'state_po', 'year']]

In [ ]:
data_1988 = pd.read_csv("../../../1988_0_0_2.csv")
data_1988 = data_1988[['FIPS', 'George Bush', 'Total Vote']]
data_1988 = data_1988[1:][:]
data_1988['share_republican'] = data_1988['George Bush'].astype(int)/data_1988['Total Vote'].astype(int)

In [ ]:
data_1992 = pd.read_csv("../../../1992_0_0_2.csv")
data_1992 = data_1992[['FIPS', 'George Bush', 'Total Vote']]
data_1992 = data_1992[1:][:]
data_1992['share_republican'] = data_1992['George Bush'].astype(int)/data_1992['Total Vote'].astype(int)

In [ ]:
data_1996 = pd.read_csv("../../../1996_0_0_2.csv")
data_1996 = data_1996[['FIPS', 'Robert Dole', 'Total Vote']]
data_1996 = data_1996[1:][:]
data_1996['share_republican'] = data_1996['Robert Dole'].astype(int)/data_1996['Total Vote'].astype(int)

In [ ]:
def format_fips(col_name, data):
    # calculate state and county fips
    data[col_name] = data[col_name].astype(int).astype(str)

    # fix one weird fips
    data.loc[[x.endswith('000') for x in data[col_name]],col_name] = [z[0:5] for z in data[[x.endswith('000') for x in data[col_name]]][col_name]]

    data.loc[:,col_name] = [x.zfill(5) for x in data[col_name]]
    data['fips_state'] = [x[0:2] for x in data[col_name]]
    data['fips_county'] = [x[2:5] for x in data[col_name]]
    data = data[['fips_state', 'fips_county', 'share_republican', 'year']]
    return(data)

In [ ]:
prez = format_fips('county_fips', prez)

In [ ]:
data_1988['year'] = 1988
data_1988 = format_fips('FIPS', data_1988)

In [ ]:
data_1992['year'] = 1992
data_1992 = format_fips('FIPS', data_1992)

In [ ]:
data_1996['year'] = 1996
data_1996 = format_fips('FIPS', data_1996)

In [ ]:
prez = pd.concat([data_1988, data_1992, data_1996, prez])

# Save presidential dataset

In [ ]:
prez.to_csv('../../../countypres_1988_2020.csv', index=False)